# Data Cleaning & Preparation

## The Foundation of Good Analysis

Clean data is crucial for accurate insights. Learn:
- Handling missing data
- Removing duplicates
- Detecting outliers
- Data normalization and scaling

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

np.random.seed(42)
sns.set_style('whitegrid')

## 1. Creating Messy Data

In [ ]:
# Create sample messy dataset
data = {
    'ID': [1, 2, 3, 3, 5, 6, 7, 8, 9, 10],
    'Name': ['Alice', 'Bob', 'Charlie', 'Charlie', 'David', 'Eve', 'Frank', 'Grace', 'Henry', 'Ivy'],
    'Age': [25, 30, np.nan, 35, 28, 32, 45, np.nan, 38, 29],
    'Salary': [50000, 60000, 55000, 55000, np.nan, 65000, 75000, 70000, 72000, 52000],
    'Bonus': [5000, np.nan, 5500, 5500, 6000, np.nan, 8000, 7000, 7200, 5200]
}

df = pd.DataFrame(data)
print("Original Messy Data:")
print(df)
print("\nData Info:")
print(df.info())

## 2. Handling Missing Values

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print("\nMissing Value Percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Method 1: Drop rows with missing values
df_dropped = df.dropna()
print("After dropping missing values:")
print(df_dropped)
print(f"Rows removed: {len(df) - len(df_dropped)}")

In [ ]:
# Method 2: Forward fill
df_ffill = df.fillna(method='ffill')
print("After forward fill:")
print(df_ffill)

In [ ]:
# Method 3: Fill with specific value
df_fillna = df.copy()
df_fillna['Age'].fillna(df_fillna['Age'].mean(), inplace=True)
df_fillna['Salary'].fillna(df_fillna['Salary'].mean(), inplace=True)
df_fillna['Bonus'].fillna(0, inplace=True)

print("After filling with mean/zero:")
print(df_fillna)

## 3. Removing Duplicates

In [ ]:
# Check for duplicates
print("Duplicates by ID:")
print(df[df.duplicated(subset=['ID'], keep=False)].sort_values('ID'))

# Remove duplicates
df_unique = df.drop_duplicates(subset=['ID'], keep='first')
print("\nAfter removing duplicates:")
print(df_unique)
print(f"Duplicates removed: {len(df) - len(df_unique)}")

## 4. Detecting Outliers

In [ ]:
# Create data with outliers
data_with_outliers = {
    'ID': range(1, 21),
    'Salary': [50000, 55000, 52000, 51000, 53000, 54000, 52000, 51000, 53000, 54000,
              52000, 51000, 53000, 54000, 52000, 51000, 53000, 54000, 500000, 52000]  # outlier at 500000
}

df_outliers = pd.DataFrame(data_with_outliers)

# IQR Method
Q1 = df_outliers['Salary'].quantile(0.25)
Q3 = df_outliers['Salary'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Bounds: [{lower_bound}, {upper_bound}]")

outliers = df_outliers[(df_outliers['Salary'] < lower_bound) | (df_outliers['Salary'] > upper_bound)]
print("\nOutliers detected:")
print(outliers)

In [ ]:
# Remove outliers
df_no_outliers = df_outliers[(df_outliers['Salary'] >= lower_bound) & (df_outliers['Salary'] <= upper_bound)]

print("After removing outliers:")
print(f"Original rows: {len(df_outliers)}, After removal: {len(df_no_outliers)}")
print(f"Min salary: {df_no_outliers['Salary'].min()}, Max salary: {df_no_outliers['Salary'].max()}")

In [ ]:
# Visualize outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
axes[0].boxplot(df_outliers['Salary'])
axes[0].set_title('Before Outlier Removal')
axes[0].set_ylabel('Salary ($)')
axes[0].grid(True, alpha=0.3)

# After
axes[1].boxplot(df_no_outliers['Salary'])
axes[1].set_title('After Outlier Removal')
axes[1].set_ylabel('Salary ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Data Normalization and Scaling

In [ ]:
# Create sample data
data_to_scale = {
    'Age': [25, 30, 35, 28, 32, 40],
    'Salary': [50000, 60000, 75000, 55000, 65000, 80000],
    'Experience': [2, 5, 10, 4, 8, 15]
}

df_scale = pd.DataFrame(data_to_scale)
print("Original Data:")
print(df_scale)
print("\nDescriptive Stats:")
print(df_scale.describe())

In [ ]:
# Standardization (Z-score normalization)
scaler_standard = StandardScaler()
df_standard = df_scale.copy()
df_standard[df_scale.columns] = scaler_standard.fit_transform(df_scale)

print("Standardized Data (Z-score):")
print(df_standard)
print("\nMean (should be ~0):", df_standard.mean())
print("Std (should be ~1):", df_standard.std())

In [ ]:
# Min-Max Scaling
scaler_minmax = MinMaxScaler()
df_minmax = df_scale.copy()
df_minmax[df_scale.columns] = scaler_minmax.fit_transform(df_scale)

print("Min-Max Scaled Data (0-1 range):")
print(df_minmax)
print("\nMin values (should be 0):", df_minmax.min())
print("Max values (should be 1):", df_minmax.max())

In [ ]:
# Visualize scaling methods
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (label, data) in enumerate([
    ('Original', df_scale),
    ('Standardized', df_standard),
    ('Min-Max', df_minmax)
]):
    axes[i].boxplot([data['Age'], data['Salary']/1000, data['Experience']*10])
    axes[i].set_title(label)
    axes[i].set_xticklabels(['Age', 'Salary (÷1000)', 'Experience (×10)'])
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Data Type Conversion

In [ ]:
# Convert data types
df_type = pd.DataFrame({
    'ID': ['1', '2', '3', '4'],
    'Amount': ['$50000', '$60000', '$55000', '$70000'],
    'Date': ['2021-01-01', '2021-02-01', '2021-03-01', '2021-04-01']
})

print("Original types:")
print(df_type.dtypes)

# Convert ID to integer
df_type['ID'] = df_type['ID'].astype(int)

# Remove $ and convert to numeric
df_type['Amount'] = df_type['Amount'].str.replace('$', '').astype(float)

# Convert to datetime
df_type['Date'] = pd.to_datetime(df_type['Date'])

print("\nConverted types:")
print(df_type.dtypes)
print("\nData:")
print(df_type)

## 📝 Exercises

### Exercise 1: Complete Data Cleaning
Apply all cleaning techniques to a messy dataset.

In [ ]:
# Sample messy data provided above
# Your task:
# 1. Handle missing values
# 2. Remove duplicates
# 3. Check for outliers
# 4. Scale the data
# 5. Display the clean dataset

## 🎓 Key Takeaways

- **Missing Data**: Drop, forward fill, or fill with mean
- **Duplicates**: Remove exact duplicates
- **Outliers**: Use IQR method to detect and remove
- **Scaling**: Standardize or normalize for machine learning
- **Data Types**: Convert to appropriate types

Next: Learn machine learning!

---
**Happy Learning! 🚀**